# 03 - SARIMA (+ Prophet secondary benchmark)

Task 3.2: `pmdarima.auto_arima` order selection on log-price (d=1) plus rolling-origin walk-forward forecasting (Listing 3.2). Run `python ../scripts/task3_point_forecasting.py` for the full, production walk-forward over the entire test block - this notebook uses a short slice so it stays interactive.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config

In [ ]:
from src.models import sarima_model

df = pd.read_parquet(config.PROCESSED_DATA_DIR / "prices_clean.parquet")
train = pd.read_parquet(config.PROCESSED_DATA_DIR / "train.parquet")
val = pd.read_parquet(config.PROCESSED_DATA_DIR / "val.parquet")
test = pd.read_parquet(config.PROCESSED_DATA_DIR / "test.parquet")
train_val = pd.concat([train, val])

In [ ]:
train_val_log_price = df.loc[train_val.index, "log_close"]
arima = sarima_model.fit_auto_arima(train_val_log_price)
print(arima.summary())

In [ ]:
sarima_model.residual_diagnostics(arima)

In [ ]:
# Short slice for a fast, interactive look; drop `.iloc[:60]` for the full test block.
test_log_price = df.loc[test.index[:60], "log_close"]
sarima_log_preds = sarima_model.walk_forward_sarima(arima, test_log_price)
sarima_price = np.exp(sarima_log_preds)

ax = df.loc[test_log_price.index, "Adj Close"].plot(label="Actual", figsize=(11, 4))
sarima_price.plot(ax=ax, label="SARIMA")
ax.legend(); ax.set_title("SARIMA walk-forward (first 60 test days)")
plt.show()

## Prophet (optional secondary benchmark, sec. 3.2)

In [ ]:
try:
    from src.models import prophet_model
    m = prophet_model.fit_prophet(train_val_log_price)
    prophet_price = np.exp(prophet_model.predict_prophet(m, test.index[:60]))
    prophet_price.plot(figsize=(11, 4), title="Prophet forecast (first 60 test days)")
    plt.show()
except ImportError as e:
    print("Prophet not installed (optional):", e)